# 02 · Kaggle metadata preprocessing

Run this notebook **first**. Attach `andyczhao/covidx-cxr2`, select version 7, and enable Internet so the two small pinned official CXR-3A label files can be downloaded. This notebook does not train a model or alter the official test partition.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/mamta-kumari28/Chest-X-Ray-Diagnosis-using-Deep-Learning-.git"
REPO_REF = "phase2/final-validation-fixes"  # Change to main after the PR is merged.
ROOT = Path("/kaggle/working/covid-xray-project")
OUTPUT_ROOT = Path("/kaggle/working/covid_xray_outputs")

if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Outputs:", OUTPUT_ROOT)

In [ ]:
# The attached Kaggle dataset must contain train/ and test/ image folders.
def find_data_root():
    candidates = [Path("/kaggle/input/covidx-cxr2")]
    candidates += sorted(Path("/kaggle/input").glob("**/train"))
    for item in candidates:
        root = item.parent if item.name == "train" else item
        if (root / "train").is_dir() and (root / "test").is_dir():
            return root
    raise FileNotFoundError("Attach Kaggle dataset andyczhao/covidx-cxr2 version 7 first.")

DATA_DIR = find_data_root()
print("Dataset:", DATA_DIR)

In [ ]:
from urllib.request import urlretrieve

LABEL_DIR = OUTPUT_ROOT / "labels_cxr3a"
PROCESSED = OUTPUT_ROOT / "processed"
LABEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

base = "https://raw.githubusercontent.com/lindawangg/COVID-Net/9645505646161c7eae5fc1a8ecbfa34a77551f2f/labels"
for name in ("train_COVIDx_CXR-3A.txt", "test_COVIDx_CXR-3A.txt"):
    target = LABEL_DIR / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
    print(name, target.stat().st_size, "bytes")

In [ ]:
METADATA = PROCESSED / "patients.csv"
if METADATA.exists():
    METADATA.unlink()  # Kaggle working output from this run only.
subprocess.run([
    sys.executable, "scripts/import_covidx_metadata.py",
    "--labels", str(LABEL_DIR / "train_COVIDx_CXR-3A.txt"), str(LABEL_DIR / "test_COVIDx_CXR-3A.txt"),
    "--data-dir", str(DATA_DIR), "--output", str(METADATA), "--require-all-images",
], check=True)

import pandas as pd
metadata = pd.read_csv(METADATA)
display(metadata.head())
display(pd.crosstab(metadata["original_partition"], metadata["label"], margins=True))
print("Saved:", METADATA)

Use **Save Version → Save & Run All**. For notebook 01, either keep the same session or attach this notebook's saved output as an input.